In [9]:
import pandas as pd

# ==============================
# LOAD DATA
# ==============================
data = pd.read_csv("../data/transactions_raw.csv")

# Convert datetime
data['timestamp'] = pd.to_datetime(data['timestamp'])

# Extract features
data['hour'] = data['timestamp'].dt.hour
data['day'] = data['timestamp'].dt.day

# ==============================
# 1. SUCCESS RATE
# ==============================
success_rate = (data['is_failed'] == 0).sum() / len(data) * 100

# ==============================
# 2. FAILURE RATE
# ==============================
failure_rate = (data['is_failed'] == 1).sum() / len(data) * 100

# ==============================
# 3. PEAK FAILURE WINDOW
# ==============================
hourly_failure = data.groupby('hour')['is_failed'].mean() * 100
peak_hours = hourly_failure.sort_values(ascending=False).head(4)

# ==============================
# 4. SALARY DAY SPIKE FACTOR
# ==============================
salary_days = data[data['day'].isin([28, 29, 30, 31])]
normal_days = data[~data['day'].isin([28, 29, 30, 31])]

salary_failure = salary_days['is_failed'].mean()
normal_failure = normal_days['is_failed'].mean()

spike_factor = salary_failure / normal_failure

# ==============================
# 5. CHANNEL RELIABILITY INDEX
# ==============================
# Total transactions
total_channel = data.groupby('payment_channel')['transaction_id'].count()

# Failure rate
failure_rate_channel = data.groupby('payment_channel')['is_failed'].mean()

# Error diversity (YOUR LOGIC)
failed_df = data[data['is_failed'] == 1]

diversity_channel = (
    failed_df.groupby('payment_channel')['failure_reason']
    .nunique()
    .reindex(total_channel.index, fill_value=0)
)

# Normalize
div_norm_channel = diversity_channel / diversity_channel.max()

# Reliability score
channel_reliability = (
    (1 - failure_rate_channel) * 70 +
    (1 - div_norm_channel) * 30
) * 100

channel_result = pd.DataFrame({
    'total_txns': total_channel,
    'failure_rate': failure_rate_channel,
    'error_diversity': diversity_channel,
    'reliability_score': channel_reliability
})

# ==============================
# 6. BANK RELIABILITY SCORE
# ==============================
total_bank = data.groupby('bank_name')['transaction_id'].count()
failure_rate_bank = data.groupby('bank_name')['is_failed'].mean()

diversity_bank = (
    failed_df.groupby('bank_name')['failure_reason']
    .nunique()
    .reindex(total_bank.index, fill_value=0)
)

div_norm_bank = diversity_bank / diversity_bank.max()

bank_reliability = (
    (1 - failure_rate_bank) * 70 +
    (1 - div_norm_bank) * 30
) * 100

bank_result = pd.DataFrame({
    'total_txns': total_bank,
    'failure_rate': failure_rate_bank,
    'error_diversity': diversity_bank,
    'reliability_score': bank_reliability
})

# Save CSV
bank_result.to_csv("reliability_scores.csv")

# ==============================
# 7. HIGH-RISK TRANSACTION RATIO
# ==============================
high_risk = data[
    (data['hour'].isin([22, 23, 0, 1])) |
    (data['day'].isin([28, 29, 30, 31]))
]

high_risk_ratio = len(high_risk) / len(data) * 100

# ==============================
# PRINT OUTPUT
# ==============================
print("\n===== KPI RESULTS =====")

print(f"\n1. Success Rate: {success_rate:.2f}%")
print(f"2. Failure Rate: {failure_rate:.2f}%")

print("\n3. Peak Failure Hours:")
print(peak_hours)

print(f"\n4. Salary Day Spike Factor: {spike_factor:.2f}x")

print("\n5. Channel Reliability Index:")
print(channel_result)

print("\n6. Bank Reliability Score:")
print(bank_result)

print(f"\n7. High-Risk Transaction Ratio: {high_risk_ratio:.2f}%")


===== KPI RESULTS =====

1. Success Rate: 76.63%
2. Failure Rate: 23.37%

3. Peak Failure Hours:
hour
1     50.000000
23    49.406176
22    48.426150
0     47.228916
Name: is_failed, dtype: float64

4. Salary Day Spike Factor: 2.25x

5. Channel Reliability Index:
                 total_txns  failure_rate  error_diversity  reliability_score
payment_channel                                                              
Card                   2017      0.206247               10        5556.271691
NEFT                   2500      0.167600               10        5826.800000
RTGS                   1010      0.091089               10        6362.376238
UPI                    3470      0.330548               10        4686.167147
Wallet                 1003      0.262213               10        5164.506481

6. Bank Reliability Score:
           total_txns  failure_rate  error_diversity  reliability_score
bank_name                                                              
Axis             